# ETL (Extract, Transform, Load) Revenue Sharing
>An ETL proccess for get data from transaction iseller and save to api endpoint landlord

Created at : 2026-04-04

Created by lahiardhan@jiwagroup.com

# IMPORT LIBRARY

In [ ]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np

import pymysql

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import paramiko

import requests
import json

In [ ]:
pd.set_option('display.float_format', '{:.4f}'.format)

# CONFIG DB

In [ ]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../db_config/config_db_datawarehouse.py")

# MAIN PROGRESS

### 1. Get Tokenization

In [ ]:
import requests
import pandas as pd


def access_token(url, username, password):

    body = {
        'grant_type' : 'password',
        'username' : username,
        'password' : password 
    }

    response = requests.post(url, body)

    if response.status_code == 200:
        token_data = response.json()
        access_token = token_data.get("access_token")
        print("Acces Token : ", access_token)
        return access_token
    else:
        print("Failed : ", response.text)
        return None


# df = pd.read_csv('get_token_stg.csv')
df = pd.read_csv('get_token.csv')

df['access_token'] = df.apply(lambda row: access_token(row['url'], row['email'], row['password']), axis=1)
df['jilid'] = df['jilid'].astype(str).str.zfill(5)

### Get Date

In [ ]:
yesterday = datetime.today().date() - timedelta(days=1)

# start_date = today
# end_date = start_date - timedelta(days=1)

start_date_str = yesterday.strftime('%Y-%m-%d')
end_date_str = yesterday.strftime('%Y-%m-%d')

# start_date_str = '2026-05-01'
# end_date_str = '2026-05-21'

print("today     :", yesterday)
print("start_day :", start_date_str)
print("end_day   :", end_date_str)

### 2. Extract Data Sales

In [ ]:
def get_sales(start_date_str, end_date_str):
    conn = mysql.connector.connect(host=config_dw.hostname,
                             port=3306,
                             user=config_dw.username,
                             password=config_dw.password,
                             db=config_dw.db_resource)
    query = f""" 
        SELECT
            DATE_FORMAT(transaction_date, '%Y-%m-%d') AS transaction_date,
            outlet_code AS jilid, 
            CASE
                WHEN transactions_payment_type_name IN (
                    'Cash',
                    'iSellerpay (Sejutajiwariders)'
                )
                THEN 'Cash'

                WHEN transactions_payment_type_name = 'iSeller Pay QRIS' THEN 'iSeller QRIS'

                ELSE transactions_payment_type_name
            END AS payment_type,
            sum(total_amount) AS total_sales,
            sum(total_tax_amount) AS total_tax,
            sum(total_amount) - sum(total_tax_amount) AS net_sales
        FROM transactions_iseller_pusat 
        WHERE DATE_FORMAT(transaction_date, '%Y-%m-%d') BETWEEN '{0}' AND '{1}'
            AND fulfillment_status = 'fulfilled'
            AND status = 'completed'
            AND payment_status = 'paid'
            AND transactions_status = 'success'
            AND outlet_code = '01065' 
        GROUP BY 1, 2, 3
        """.format(start_date_str, end_date_str)

    df_sales = pd.read_sql(query,conn)
    conn.close()
    return df_sales 

sales = get_sales(start_date_str, end_date_str)
# final_data = sales.groupby(['jilid', 'transaction_date', 'payment_type'])({'total_amount': 'sum', 'total_tax': 'sum', 'net_sales': 'sum'}).reset_index()

In [ ]:
final_data = sales.groupby(
    ['jilid', 'transaction_date', 'payment_type']
).agg({
    'total_sales': 'sum',
    'total_tax': 'sum',
    'net_sales': 'sum',
}).reset_index()

# TransactionNumber harus dari baris final_data (sudah aggregate), bukan dari sales (banyak baris)
import re

def _txn_token(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"[^\w\-]+", "_", s, flags=re.UNICODE)  # ganti karakter “aneh” jadi _
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "UNKNOWN"
pt = final_data["payment_type"].map(_txn_token)
final_data["TransactionNumber"] = (
    final_data["jilid"].astype(str)
    + "-"
    + final_data["transaction_date"].astype(str)
    + "-"
    + pt
    + "-"
    + pd.Timestamp.now().strftime("%H%M%S")
)
final_data.columns = ['jilid', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount', 'TransactionNumber']
final_data['TransactionNumber'] = 'Janji_Jiwa_' + final_data['jilid'] + '_' + final_data['PaymentType'].astype(str) + '_' + final_data["TransactionDate"].astype(str).str.replace('-', '', regex=False) + '_' + pd.Timestamp.now().strftime("%H%M%S")
# Count sharing amount to landlord 
final_data['Amount'] = final_data['Amount'].round(4)
# final_data['Remarks'] = 'Janji Jiwa ' + final_data['jilid'] + ' ' + final_data['PaymentType'].astype(str) + ' ' + final_data["TransactionDate"].astype(str) + '-' + pd.Timestamp.now().strftime("%H%M%S")
final_data['Remarks'] = 'Janji_Jiwa_' + final_data['jilid'] + '_' + final_data['PaymentType'].astype(str) + '_' + final_data["TransactionDate"].astype(str) + '_' + pd.Timestamp.now().strftime("%H%M%S")

final_data

In [ ]:
# print(final_data['Amount'].sum())
# print(final_data['Amount'].sum() * 0.1)
# print(final_data['Sharing'].sum())

### 3. Transform and Load Data to API

In [ ]:
merged = pd.merge(final_data, df, on='jilid')
merged['url'] = 'https://tenant.lippomallpuri.com/revenue/api/revenue/v1/save'

all_log_summary = []
all_log_detail = []

for jilid, group in merged.groupby('jilid'):
    url = group['url'].iloc[0]
    token = group['access_token'].iloc[0]
    revenue_data = group[['TransactionNumber', 'TransactionDate', 'Amount', 'Remarks']].to_dict('records')

    request_payload = {"revenueDatas": revenue_data}

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    response = requests.post(url, json=request_payload, headers=headers)
    now = datetime.datetime.now()

    if response.status_code == 200:
        try:
            resp_json = response.json()
        except ValueError:
            resp_json = {
                'Message': f'Non-JSON 200 body: {response.text[:500]}',
                'transactionReturnDatas': [],
                'ErrorTransactionNumber': [],
            }

        log_summary = pd.DataFrame([{
            'revenue_batch_id': resp_json.get('RevenueBatchId'),
            'jilid': jilid,
            'transaction_date': group['TransactionDate'].iloc[0],
            'total_records_sent': len(revenue_data),
            'total_success': resp_json.get('TotalSuccess'),
            'total_error': resp_json.get('TotalError'),
            'http_status_code': response.status_code,
            'api_code': resp_json.get('Code'),
            'api_message': resp_json.get('Message'),
            'api_url': url,
            'request_payload': json.dumps(request_payload),
            'response_payload': json.dumps(resp_json),
            'created_at': now
        }])
        all_log_summary.append(log_summary)

        request_df = group[['jilid', 'TransactionNumber', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount', 'Remarks']].copy()
        request_df.columns = ['jilid', 'transaction_number', 'transaction_date', 'payment_type', 'total_sales', 'total_tax', 'amount', 'remarks']

        response_details = pd.DataFrame(resp_json.get('transactionReturnDatas', []))
        if not response_details.empty:
            response_details = response_details.rename(columns={
                'TenantTransactionNumber': 'transaction_number',
                'MallTransactionId': 'mall_transaction_id'
            })
            detail = request_df.merge(response_details, on='transaction_number', how='left')
        else:
            detail = request_df.copy()
            detail['mall_transaction_id'] = None

        error_txns = resp_json.get('ErrorTransactionNumber', [])
        error_map = {}
        for err in error_txns:
            if '|' in str(err):
                txn_num, error_msg = str(err).split('|', 1)
                error_map[txn_num] = error_msg
            else:
                error_map[str(err)] = 'unknown error'

        detail['revenue_batch_id'] = resp_json.get('RevenueBatchId')
        detail['status'] = detail['mall_transaction_id'].apply(lambda x: 'success' if pd.notna(x) else 'error')
        detail['error_message'] = detail['transaction_number'].map(error_map)
        detail['created_at'] = now
        all_log_detail.append(detail)

        print(f"SUCCESS | jilid: {jilid} | {response.status_code} | {resp_json.get('Message')}")
    else:
        log_summary = pd.DataFrame([{
            'revenue_batch_id': None,
            'jilid': jilid,
            'transaction_date': group['TransactionDate'].iloc[0],
            'total_records_sent': len(revenue_data),
            'total_success': 0,
            'total_error': len(revenue_data),
            'http_status_code': response.status_code,
            'api_code': None,
            'api_message': response.text[:500],
            'api_url': url,
            'request_payload': json.dumps(request_payload),
            'response_payload': response.text,
            'created_at': now
        }])
        all_log_summary.append(log_summary)

        detail = group[['jilid', 'TransactionNumber', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount', 'Remarks']].copy()
        detail.columns = ['jilid', 'transaction_number', 'transaction_date', 'payment_type', 'total_sales', 'total_tax','amount', 'remarks']
        detail['mall_transaction_id'] = None
        detail['revenue_batch_id'] = None
        detail['status'] = 'http_error'
        detail['error_message'] = f"HTTP {response.status_code}: {response.text[:200]}"
        detail['created_at'] = now
        all_log_detail.append(detail)

        print(f"FAILED | jilid: {jilid} | {response.status_code} | {response.text}")

df_log_summary = pd.concat(all_log_summary, ignore_index=True) if all_log_summary else pd.DataFrame()
df_log_detail = pd.concat(all_log_detail, ignore_index=True) if all_log_detail else pd.DataFrame()

print(f"\nTotal log summary: {len(df_log_summary)} rows")
print(f"Total log detail: {len(df_log_detail)} rows")

In [ ]:
df_log_summary

In [ ]:
df_log_detail

### 4. Dump Log to DB (api_landlord_log + api_landlord_log_detail)

In [ ]:
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(
    hostsname=config_dw.hostname,
    user=config_dw.username,
    pw=config_dw.password,
    db=config_dw.db_resource
))

In [ ]:
if not df_log_summary.empty:
    df_log_summary.to_sql('api_landlord_logs', con=engine, if_exists='append', index=False)
    print(f"api_landlord_log: {len(df_log_summary)} rows inserted")
else:
    print("api_landlord_log: no data to insert")

In [ ]:
df_log_detail

In [ ]:
if not df_log_detail.empty:
    df_log_detail.to_sql('api_landlord_log_detail', con=engine, if_exists='append', index=False)
    print(f"api_landlord_log_detail: {len(df_log_detail)} rows inserted")
else:
    print("api_landlord_log_detail: no data to insert")

### 5. Send Email to Data Team

> Cells below are for sending notification email

In [ ]:
import os
import smtplib, ssl

port = 587  # For starttls
smtp_server = "smtp.gmail.com"
sender_email = "saputra.christabel20@gmail.com"
receiver_email = ["athens.jiwagroup@gmail.com", "lahia.ardhanlahia@gmail.com"]
password = os.getenv("GMAIL_SMTP_PASSWORD")

message = """Subject: [NEW] - Send API Landlord Yesterday - Done

Hi ! This is reporting Data for You !

This message is sent from Jupyter Notebook.

Best Regards,
DatO ( Data autOmation )"""

context = ssl.create_default_context()
with smtplib.SMTP(smtp_server, port) as server:
    server.ehlo()  # Can be omitted
    server.starttls(context=context)
    server.ehlo()  # Can be omitted
    server.login(sender_email, password)
    server.sendmail(sender_email, receiver_email, message)